Try the method of constructing a good template through a combination of existing.

Steps:
- Pick (random select) a BTS non-Ia SN and retrieve the S params (and color).
- Load all (good) sncosmo models of the correct type, and their S params.
- Select a subset of these to serve as base. Selection can be random, or based on how well the globally seems to match BTS SNe (such that weird templates get selected less often).
- - Go with three, as these means solution determined. 
- Find the coefficients which make the base models reproduce the observed (+ scatter).
- - linalg.solve(A,b)
- https://meshlogic.github.io/posts/jupyter/linear-algebra/linear-algebra-numpy-2/
- For other https://eng.libretexts.org/Courses/Arkansas_Tech_University/Engineering_Modeling_and_Analysis_with_Python/14%3A_Linear_Algebra_Equations/14.04%3A_Overdetermined_Systems
- Overplot to see whether the lightcurves look ok (possibly have to include scaling + color).

Seems like we cannot quite get this to work. Some weird narrow sne cannot be recreated through whatever combos we attempt. Maybe not surprising since parsnip models are not linear, so no appriory reason that adding two models together will create whatever lies in that parsnip space. Try something else?


In [ ]:
import pymongo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sncosmo
import pickle
import json
import parsnip
import lcdata
import astropy
from ampel.ztf.util.ZTFIdMapper import ZTFIdMapper
from ampel.ztf.view.ZTFFPTabulator import ZTFFPTabulator 

In [ ]:
# Parameters
snix = 2
scatter_sigma = 0.1   # How much should we shift positions in parsnip space? 
parsnip_disperson = 0.0  # Additional scatter for more variability - not sure needed
include_sigma = 5.  # Reject outlying datapoints

In [ ]:
# Semi permanent
# Input supernova data
df_sn = pd.read_csv('/Users/jnordin/tmp/bts_poormatch_goodlc.csv')
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')
# parsnip
ppath = '/Users/jnordin/data/models/parsnip/v3_Jul2025/noiztf_model_hybrid_snia_scale2_10_unbalanced_z0_sncut_subsamp0.9_nocadscale_seed0.pt'

### 1. Retrieve a SN and its class, inspect parsnip fit. 

In [ ]:
# Seems like we now subsample from a list of poor fit objects, extend later
sndata = dict(df_sn.iloc[snix])
z = float(list(df_bts[df_bts['ZTFID']==sndata['mod']]['redshift'])[0])
sndata

#### Get photometric (baseline corrected) from db

In [ ]:
tabulators = []
tabulators.append(
    ZTFFPTabulator(inclusion_sigma=include_sigma)    
)
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts
def get_db_table( name, database, tabulators=[] ):
    """
    For ZTF name, get photopoints and then tables. 
    """

    # Name
    if isinstance(name, int):    
        print('Assuming name given as stock')
        stock = int
    elif re.search('ZTF', name):
        stock = ZTFIdMapper.to_ampel_id(name)
    else:
        print('Cannot parse', name)
        return None 

    # Obtain photopoints
    dps = [dp for dp in database.t0.find({'stock':stock})]

    # Convert to table(s)
    ftables = []
    for tabulator in tabulators:
        ftables.append( tabulator.get_flux_table(dps) )
    if len(ftables)>1:
        print('Implement astropy table appending!')
    return ftables.pop(0)

In [ ]:
tab = get_db_table(sndata['mod'], database=db, tabulators=tabulators)
tab.sort('time')

In [ ]:
tab.meta = {
            'object_id':sndata['mod'],
            'type':sndata['class'],
            'redshift':z,
        }

#### Do parsnip fit 

In [ ]:
model = parsnip.load_model(ppath)

In [ ]:
mfit = model.predict_light_curve(tab)

In [ ]:
parsnip.plot_light_curve(tab, model)

Looking good? Inspect pulls? Anyway, if you feel model is good enough, continue

### Find three template models to compare with

In [ ]:
dft = pd.read_csv('/Users/jnordin/data/models/parsnip/v3_Jul2025/modelfits/sncosmotemplates_parsnip_ampcol.csv')

In [ ]:
# We do need some sncosmo template matching
sncosmo_modelmap = {
     'PopIII':'PopIII',
     'SN II':'SN II',
     'SN II-pec':'SN II',
     'SN IIL':'SN II',
     'SN IIL/P':'SN II',
     'SN IIP':'SN II',
     'SN IIb':'SN II',
     'SN IIn':'SN II',
     'SN Ia':'SN Ia',
     'SN Ib':'SN Ib/c',
     'SN Ib/c':'SN Ib/c',
     'SN Ic':'SN Ib/c',
     'SN Ic-BL': 'SN Ib/c',
    'SN Icn': 'SN Ib/c', 
    'SN Ia-CSM': 'SN Ia', 
    'SN Ic-pec': 'SN Ib/c', 
    'SN Ia-91bg': 'SN Ia', 
    'SN Ia-91T': 'SN Ia', 
    'SN Ca-rich-Ca': None, # No templates
    'SN Ibn': 'SN Ib/c', 
    'Ca-rich': None,
    'SN Ib/c': 'SN Ib/c', 
    'SN II-pec': 'SN II', 
    'ILRT': None,      # No templates 
    'SN Ia-pec': 'SN Ia', 
    'TDE': None,       # No templates 
    'SN Iax': 'SN Ia', 
    'SLSN-I': 'SN II',      # No templates available, using these for now
    'SN Ia-SC': 'SN Ia', 
    'SN IIn': 'SN II', 
    'SN IIn-pec': 'SN II', 
    'SN Ib-pec': 'SN Ib/c',
}

In [ ]:
dft['wideclass'] = dft['class'].map(sncosmo_modelmap)

In [ ]:
# Subselect to templates of the same 
if sum(dft['class']==sndata['class'])>0:
    print('Template exists for specific class - use this')
    dfclass = dft[dft['class']==sndata['class']]
else:
    print('No specific class, using wide class')
    dfclass = dft[dft['wideclass']==sndata['wideclass']]

In [ ]:
#dfclass = dft[dft['wideclass']==sndata['wideclass']]

In [ ]:
#dfclass = dft

In [ ]:
dfclass

In [ ]:
# Create taret hyperparameter position

In [ ]:
N1 = sndata['S1'] + sndata['dS1'] * np.random.normal(size=1, loc=0, scale=scatter_sigma) + np.random.normal(size=1, loc=0, scale=parsnip_disperson)
N2 = sndata['S2'] + sndata['dS2'] * np.random.normal(size=1, loc=0, scale=scatter_sigma) + np.random.normal(size=1, loc=0, scale=parsnip_disperson)
N3 = sndata['S3'] + sndata['dS3'] * np.random.normal(size=1, loc=0, scale=scatter_sigma) + np.random.normal(size=1, loc=0, scale=parsnip_disperson)

In [ ]:
print(sndata['S1'], sndata['S2'], sndata['S3'])

In [ ]:
print(N1,N2,N3)

In [ ]:
# Geometric difference
gdiff = (
            (N1-dfclass['S1'])**2 / (sndata['dS1']**2+dfclass['dS1']**2) + 
            (N2-dfclass['S2'])**2 / (sndata['dS2']**2+dfclass['dS2']**2) + 
            (N3-dfclass['S3'])**2 / (sndata['dS3']**2+dfclass['dS3']**2)  
)

In [ ]:
# Invert to relative properties
rprop = 1 / gdiff
rprop /= np.sum(rprop)

In [ ]:
# We now wish to select X (3?) objects from these, using these
# as some relative prop. What if we draw a uniform each and then multiply?
rprop

In [ ]:
rand = np.random.uniform(size=len(rprop))

In [ ]:
sel = rprop * rand

In [ ]:
sel

In [ ]:
sel.nlargest(n=3)

In [ ]:
im = list( sel.nlargest(n=3).index )

In [ ]:
im

In [ ]:
basemodels = dft.loc[im]

In [ ]:
basemodels

#### Find the linear coefficients required 

In [ ]:
b = np.matrix( [N1,N2,N3] )

In [ ]:
A = np.matrix( basemodels[['S1','S2','S3']] ).T

In [ ]:
A

In [ ]:
np.linalg.matrix_rank(A)

In [ ]:
x = np.linalg.solve(A,b)

In [ ]:
x

In [ ]:
weights = np.array(x)

In [ ]:
basemodels['amplitude']

### Try an overdetermined system 

In [ ]:
im2 = list( sel.nlargest(n=10).index )

In [ ]:
basemodels2 = dft.loc[im2]

In [ ]:
A2 = np.matrix( basemodels2[['S1','S2','S3']] ).T

In [ ]:
x2, res, rank, s = np.linalg.lstsq(A2, b,rcond=None)

In [ ]:
x2

In [ ]:
weights2 = np.array( x2 )

#### Try to get positive coefficients through NonNegative Least Squares 

In [ ]:
from scipy.optimize import nnls

In [ ]:
im2 = list( sel.nlargest(n=5).index )

In [ ]:
basemodels2 = dft.loc[im2]

In [ ]:
A2 = np.matrix( basemodels2[['S1','S2','S3']] ).T

In [ ]:
(x2, rnorm) = nnls(A2, np.array([N1[0],N2[0],N3[0]]),maxiter=100000)

In [ ]:
x2

In [ ]:
weights2 = x2

In [ ]:
basemodels2['amplitude']

In [ ]:
weights2

In [ ]:
list(basemodels['amplitude'])

### Create Frankesteins model. 

In [ ]:
class FrankensteinSource(sncosmo.Source):

    _param_names = ['amplitude']
    param_names_latex = ['A']   # used in plotting display

    def __init__(self, 
                    sources: list[str],
                    weights: list[float],
                    source_amplitudes: list[float] | None = None,
                    source_colors: list[float] | None = None,
                    peak_source_bessellb: float | None = None,
                    inv: bool = False,
                    name: str | None = None, version: str | None = None):
        self.name = name
        self.version = version

        # Do this through sources 
        #self.sources = [
        #    sncosmo.get_source(source) for source in sources
        #]
        # What about individual amplitudes? Assumed they were all equal?
        #if source_amplitudes:
        #    for s,a in zip(self.sources, source_amplitudes):
        #        s.set(amplitude=a)            
        # Set to common thing?
        # Or do we have to retain amplitude got when fitting? 
#        if peak_source_bessellb:
#            for s in self.sources:
#                s.set_peakmag(peak_source_bessellb, 'bessellb', 'ab')

        # Alternatively, use models
        # More complex and unintuitive, but allows color etc.
        
        self.sources = []
        for source in sources:
            model = sncosmo.Model(source=source)
            if peak_source_bessellb:
                model.source.set_peakmag(peak_source_bessellb, 'bessellb', 'ab')
            if source_amplitudes:
                if inv:
                    model.set(amplitude = 1. / source_amplitudes.pop())
                else:
                    model.set(amplitude = source_amplitudes.pop())
            if source_colors:
                model.add_effect( sncosmo.CCM89Dust(), 'host', 'rest' )
                model.set(hostr_v=3.1)
                if inv:
                    model.set(hostebv=source_colors.pop())
                else:
                    model.set(hostebv=-source_colors.pop())
            self.sources.append( model )

        
        self.weights = weights
        print(len(self.sources))
        print(len(self.weights))
        # The idea assumes similar amplitudes? Check this?
        if not len(self.sources)==len(self.weights):
            sys.exit('Require one weight per source!')
        # Potentially, here check that the combination makes sense, i.e. not too negative.
        # Possibly run through all models and see which makes sense, i.e. provides
        # mathches that work. 
            
        self._parameters = np.array([1.])  # initial parameters

        # We need to provide max and min wavelength and phase.
        # Either overrride methods to check templates, 
        # create dummpy phase and wave variables with two entries
        # or really interpolate what it should be 
        self._phase = [
            max([source.source.minphase() for source in self.sources]),
            min([source.source.maxphase() for source in self.sources]) 
           ]
        self._wave = [
            max([source.source.minwave() for source in self.sources]),
            min([source.source.maxwave() for source in self.sources]) 
           ]

    def _flux(self, phase, wave):
        amplitude = self._parameters

        totflux = sum( [
            w*submod.flux(phase,wave) for submod, w in zip( 
                self.sources,
                self.weights
            )
        ] )
        # Ensure dimension compatible with single source 
        if totflux.ndim==0:
            totflux = np.expand_dims(totflux, axis=(0,1))
        elif totflux.ndim==1:
            totflux = np.expand_dims(totflux, axis=0)
        
        return amplitude * totflux


In [ ]:
fs = FrankensteinSource(sources=basemodels['mod'], weights=weights)

In [ ]:
fs = FrankensteinSource(sources=basemodels['mod'], 
                        weights=weights,
                       source_amplitudes=list(basemodels['amplitude'])
                       )

In [ ]:
fs = FrankensteinSource(sources=basemodels['mod'], 
                        weights=weights,
                        source_amplitudes=list(basemodels['amplitude']), 
                        source_colors=list(basemodels['color'])
                       )

In [ ]:
fs = FrankensteinSource(sources=basemodels2['mod'], 
                        weights=weights2,
#                        source_amplitudes=list(basemodels2['amplitude']), 
                        source_colors=list(basemodels2['color']), 
#                        peak_source_bessellb=-19, 
                        inv=True
                       )

In [ ]:
from matplotlib import pyplot as plt

wave = np.linspace(3000.0, 10000.0, 500)
for p in (-10,-5,0,5,10,20):
    plt.plot(wave, fs.flux(p, wave), label='w={:3.1f}'.format(p))

plt.legend()
plt.show()


In [ ]:
phases = np.linspace(-10,30,10)
for band in ('ztfg','ztfr','ztfi'):
    plt.plot(phases, fs.bandflux(band, phases), 'o', label=band)

plt.legend()
plt.show()


In [ ]:
dust = sncosmo.CCM89Dust()

In [ ]:
model = sncosmo.Model(source=fs,
                      effects=[dust],
                      effect_names=['host'],
                      effect_frames=['rest'])

In [ ]:
model.param_names

In [ ]:
model.set(z=z)
model.set(hostr_v=3.1)

In [ ]:
# run the fit
result, fitted_model = sncosmo.fit_lc(
    tab, model,
    ['t0', 'amplitude', 'hostebv'],  # parameters of model to vary
#    bounds={'hostebv':(-0.5, 0.5)}  # bounds on parameters (if any)
)

In [ ]:
print("Number of chi^2 function calls:", result.ncall)
print("Number of degrees of freedom in fit:", result.ndof)
print("chi^2 value at minimum:", result.chisq)
print("model parameters:", result.param_names)
print("best-fit values:", result.parameters)
print("The result contains the following attributes:\n", result.keys())

In [ ]:
fig = sncosmo.plot_lc(tab, model=fitted_model, errors=result.errors)

# Ok, so to test this, what are the variables?
- Use positive linear fit, or exact? Prelim seems to indicate the first.
- Propagate amplitude or inverse amplitude from the template fit to the model?
- Propagate color or neg color from the template fit?
- Add an absolute magnitude scale (and how to combine with amplitude/color vals).
- How many components (for nnls)
- Whether to cut out some templates that only makes things worse.
- Whether to use narrow, wide or any classes.
- Fit individual weight?
- 